# AI Sommelier RAG

In [6]:
from dotenv import load_dotenv
load_dotenv()

True

#### 1. 요리 풍미 묘사 (Image Analysis)

In [7]:
from langchain_core.prompts import ChatPromptTemplate, HumanMessagePromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

def describe_dish_flavor(query):
    # 프롬프트 정의
    prompt = ChatPromptTemplate.from_messages([
        # 키워드의 순서보다는 키워드 자체가 중요함.
        # system : 시스템 메세지 (역할 부여, 페르소나 등)
        # human(user) : 사용자가 입력한 질문
        # ai(assistant) : AI의 대답(대화 기록을 넣을 때 주로 사용)
        ('system',"""
        Persona: You are a highly skilled food expert with a deep understanding of culinary techniques, flavor profiles, and ingredient pairings. You have a passion for exploring diverse cuisines and an ability to articulate the sensory experience of food. Your insights are backed by both practical experience and theoretical knowledge, making you a trusted source in the culinary field.

        Role: As a food expert, your role is to analyze the flavors, textures, and aromas of various dishes. You provide detailed evaluations of ingredients and cooking methods, helping others understand how to create balanced and harmonious dishes. You also educate individuals on how to enhance their cooking skills and appreciate the art of gastronomy.

        Examples:

        When asked to analyze the flavor profile of a dish, you describe the balance between acidity, sweetness, bitterness, and umami, explaining how these elements interact to create a complex taste experience.
        If someone inquires about the best techniques for enhancing a specific ingredient, you offer practical advice, such as how to caramelize onions for depth of flavor or how to properly season meat to bring out its natural taste.
        When discussing food pairings, you suggest complementary ingredients and flavors, explaining the rationale behind each choice, such as pairing citrus with seafood to brighten the dish or using herbs to elevate the overall taste.
        """),
        ('user', '이미지의 요리명과 풍미를 한 문장으로 요약해주세요.')
    ])
    # 이미지 URL 메시지 추가
    prompt += HumanMessagePromptTemplate.from_template([query])

    # 모델 설정
    model = ChatOpenAI(model='gpt-4o', temperature=0)

    # output parser
    output_parser = StrOutputParser()

    # 체인 설정
    chain = prompt | model | output_parser
    return chain

In [8]:
from langchain_core.runnables import RunnableLambda

chain = RunnableLambda(describe_dish_flavor)
chain.invoke({'image_url':'https://media-cdn.tripadvisor.com/media/photo-s/09/9a/02/da/caption.jpg'})

'이 요리는 바비큐 립으로, 풍미는 달콤하고 짭짤한 소스가 어우러져 고기의 풍부한 맛을 강조하며, 곁들여진 코울슬로가 상큼함을 더해줍니다.'

#### 2. 리뷰에서 와인 검색(Retrieval)

In [12]:
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore
import os

# 환경변수 로드
PINECONE_INDEX_NAME = os.getenv('PINECONE_INDEX_NAME')
PINECONE_NAMESPACE = os.getenv('PINECONE_NAMESPACE')
PINECONE_API_KEY = os.getenv('PINECONE_API_KEY')

def search_wines(query):
    """
    Args:
        query(str): 요리 풍미 묘사 텍스트
    Return:
        dict: 다음 단계로 넘겨줄 데이터 셋 (요리 묘사 + 검색된 와인 리뷰)
    """

    embeddings = OpenAIEmbeddings(model='text-embedding-3-small')
    
    # Pinecone DB연결
    vector_db = PineconeVectorStore(
        embedding=embeddings, # 사용할 임베딩 모델
        index_name=PINECONE_INDEX_NAME,
        namespace=PINECONE_NAMESPACE,    # 데이터를 구분 보관할 네임스페이스
        pinecone_api_key=PINECONE_API_KEY
    )

    #유사도 검색
    results = vector_db.similarity_search(
        query,
        k=5,
        namespace=PINECONE_NAMESPACE
    )

    # dish_flavor: 원래 요리 설명 (그대로 전달)
    # wine_reviews: 검색된 와인 리뷰들을 하나의 문자열로 합침
    return {"dish_flavor": query,
            "wine_reviews": "\n".join([doc.page_content for doc in results])}


In [ ]:
chain = RunnableLambda(search_wines)
chain.invoke('이 요리는 바비큐 립으로, 풍부한 훈제 향과 달콤하고 짭짤한 소스가 어우러져 깊은 풍미를 자아냅니다.')

{'dish_flavor': '이 요리는 바비큐 립으로, 풍부한 훈제 향과 달콤하고 짭짤한 소스가 어우러져 깊은 풍미를 자아냅니다.',
 'wine_reviews': ": 85863\ncountry: US\ndescription: This kitchen sink-style blend shows a whiff of volatility along with notes of vanilla and cherry candy. The palate is full of sweet, plump fruit flavors that linger on the finish.\ndesignation: Oink!\npoints: 87\nprice: 15.0\nprovince: America\nregion_1: \nregion_2: \ntaster_name: Sean P. Sullivan\ntaster_twitter_handle: @wawinereport\ntitle: BBQ Wine Company NV Oink! Red (America)\nvariety: Red Blend\nwinery: BBQ Wine Company\n: 72759\ncountry: US\ndescription: A “best-barrel blend” that's a 50-50 Pommard and Dijon clone mix. The black-cherry fruit is framed with a bit more barrel toast than the other Brick House Pinots, having been aged for 18 months in 40% new wood. As with all BH wines, it's impeccably clean, minimally handled and elegantly balanced. Biodynamic farming brings added textural details, lengthens the finish and surprises with its complexity.\

#### 3. 최종 와인 추천 (Generation)

In [ ]:
def recommend_wines(query):
    # 프롬프트 정의
    prompt = ChatPromptTemplate.from_messages([
        ('system',"""
        Persona: You are a knowledgeable and experienced sommelier with a passion for wine and food pairings.
        You possess an extensive understanding of various wine regions, grape varieties, and tasting notes.
        Your demeanor is friendly and approachable.

        Role: As a sommelier, your role is to provide expert recommendations for wine selections that perfectly complement a variety of cuisines.

        Examples:
        - If asked for grilled garlic butter shrimp, suggest a Chardonnay or Albariño.
        - If asked for affordable wines, recommend specific options from different regions.
        """),
        HumanMessagePromptTemplate.from_template("""
        와인페어링 추천에 아래의 요리와 풍미, 와인리뷰만을 참고하여 한글로 답변해주세요.
        
        요리와 풍미:
        {dish_flavor}

        와인리뷰:
        {wine_reviews}
        """)
    ])
    
    model = ChatOpenAI(model='gpt-5-nano', temperature=1)

    chain = prompt | model | StrOutputParser()

    return chain

#### 전체 파이프라인 연결 (LCEL)

In [18]:
from langchain_core.runnables import RunnableLambda

chain = RunnableLambda(describe_dish_flavor) | RunnableLambda(search_wines) | RunnableLambda(recommend_wines)

# 실행
image_url='https://semie.cooking/image/contents/recipe/lm/bg/rfifbtxa/127147885dkig.jpg'

response = chain.invoke({'image_url':image_url})

print(response)

다음은 떡볶이의 매콤달콤한 풍미와 주어진 와인리뷰를 바탕으로 한 추천입니다.

- Domaine Ehrhart 2006 Herrenweg Riesling (Alsace)
  - 왜 어울리나요: 달콤한 과일향과 잔잔한 산미가 결합된 리슬링으로, 잔맛이 길고 달콤함의 균형이 잘 잡혀 있습니다. 리뷰에는 apricot와 wet stone 노트와 잔당이 남는 느낌이 언급되어 있어 매콤한 소스의 단맛과 잘 어울리며, 아시아 퓨전 스타일 요리와의 페어링 제안도 있습니다.
  - 요약 포인트: 달콤함+산미의 균형이 떡볶이의 매운맛을 상쇄하고 풍미를 길게 남깁니다.
  - 가격대: 약 22.0

- Ochoa 2010 Vino Dulce Moscatel (Navarra, Spain)
  - 왜 어울리나요: 오렌지 마말레이드, 꿀, 복숭아 같은 달콤한 과일향에 산도가 있어 단맛이 너무 뚜렷하지 않고 균형감을 제공합니다. 매콤달콤한 소스의 단맛과의 조화가 돋보일 수 있습니다.
  - 요약 포인트: 달콤함이 강조되지만 산미가 남아 무거움 없이 길게 마무리됩니다.
  - 점수/가격: 92점, 약 28.0

- Chateau Morrisette 2010 Frosty Dog White (Virginia)
  - 왜 어울리나요: 달콤한 스타일이 흔히 매콤한 요리와 잘 어울리는 산도와 가벼운 바디감을 갖추고 있으며, exotic fruit와 flower petal 향이 떡볶이의 풍미와도 색다르게 어울립니다. 리뷰에서 느낄 수 있는 밝고 경쾌한 맛이 매운맛을 상쇄하는 데 도움이 될 수 있습니다.
  - 요약 포인트: 산도와 달콤함의 균형이 가볍고 산뜻하게 마무리되어 매운맛 뒤에 깔끔한 여운을 남깁니다.
  - 가격대: 약 20.0

참고로, 이 목록 중에서 매운맛이 강한 떡볶이와의 매칭을 더 적극적으로 원하신다면, 과일향과 산미가 돋보이는 Riesling 계열이 가장 안정적이고, 그다음으로 달콤한 MOSCATEL 계열이 달콤함으로 매운맛의 강도를 상쇄하는 방향으로 흐르는 경향이 있습니다. 